# F02-P2 Nature

**Site Characterization: Nature, components 2.1 and 2.2.**

Characterizes the ecological quality of the AOI forest and habitat. Where `F02-P2 General`
answers "where is this site and what has happened to it", Nature answers "how good is what is
left, and does it matter for biodiversity".

> **Not runnable yet.** All analysis logic below is real Python. Only file access is stubbed.

**Status.** Complete as scoped: 2.1 FLII and 2.2 KBA.

**Known limit of that scope.** Both components need forest or a designated site to say anything. 2.1
FLII is a property of forest and returns not applicable without it, and 2.2 KBA returns a negative
sentence on most sites. A degraded grassland or a cropland targeted for planting therefore receives
almost no ecological description from this module, which is the site type where a RESTORE decision
most depends on the starting condition. Related asymmetry: the reference ecosystem has five classes
in the backend, but the only ecosystem quality metric here measures forest, so savanna and grassland
have no equivalent of FLII. Recorded as a scope limit, not as pending work.

## Handoff

Reads nothing from `F02-P2 General` at present. Both Nature components derive their own forest
extent from `forest_mask_2024`, so this notebook can run independently. It writes
`outputs/<aoi_id>__F02-P2-nature.json`.

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

from dataclasses import dataclass

import geopandas as gpd
import numpy as np

from config import *
from common import *

In [ ]:
AOI_PATH = r"<SET: path to the project AOI polygon>"
aoi_id = "<SET: short id for this run, must match F02-P2 General>"

aoi = prepare_aoi(gpd.read_file(AOI_PATH))
print(f"AOI {aoi_id}: {fmt_ha(aoi.area_ha)}")

---
## 2.1 Forest Landscape Integrity (FLII)

Reports the landscape integrity of the AOI forest as a headline mean score out of 10, with a
High / Medium / Low breakdown in the narrative. Integrity is the degree to which a forest is
still intact, connected, and free of human pressure.

**Data.** `flii_forest_mosaic_SEA_300m.tif` (continuous 0 to 10, on forest) and
`flii_class_mosaic_SEA_300m.tif` (1 = Low, 2 = Medium, 3 = High, on forest). Based on the Forest
Landscape Integrity Index (Grantham et al. 2020, Nat. Commun. 11:5978), reimplemented and
calibrated on the SEA data stack. Landscape scale, native 300 m.

**Calibration warning.** The 0 to 10 values are calibrated on the pooled SEA distribution, so
they are not one to one with the published global FLII product. Present them as the SEA forest
integrity layer, internally consistent within this run. Do not claim absolute global integrity.
Class breaks follow the paper: High at or above 9.6, Low at or below 6.0, Medium in between.

**Example render (predominantly High).**

> **Forest landscape integrity: 8.8 / 10**
>
> Of the forest in this area, 68% has high landscape integrity, 24% medium, and 8% low. The
> forest is predominantly high integrity, indicating largely intact and well-connected forest
> under low human pressure.

**Downstream use.** FLII is a biodiversity and ecosystem quality proxy feeding Triple Win
Pillar 1, and a pathway signal: high integrity favours PROTECT, low integrity favours RESTORE or
MANAGE. It also underpins the SCeNe high-integrity NbS criteria.

In [ ]:
FLII_LOW, FLII_MEDIUM, FLII_HIGH = 1, 2, 3

FLII_GLOSS = {
    FLII_HIGH: "indicating largely intact and well-connected forest under low human pressure",
    FLII_MEDIUM: (
        "indicating moderately modified forest with some fragmentation or human pressure"
    ),
    FLII_LOW: "indicating heavily modified and fragmented forest under high human pressure",
}


def analyze_flii(aoi: AOI) -> ComponentResult:
    """Component 2.1. Forest landscape integrity over the AOI forest."""
    # The FLII rasters are already masked to forest upstream, so their valid extent defines the
    # forest here. forest_mask_2024 is loaded only to catch the "no forest at all" case early,
    # so the message matches 1.5 and 1.6 rather than saying "no FLII data".
    if forest_mask_2024(aoi).is_empty:
        return not_applicable(
            "2.1 Forest Landscape Integrity",
            "No forest is present in this project area, so landscape integrity cannot be "
            "assessed.",
        )

    score = load_raster_clipped(FLII_FOREST_RASTER, aoi, resampling="bilinear")
    classes = load_raster_clipped(FLII_CLASS_RASTER, aoi, resampling="nearest")

    forest_area_ha = classes.valid_area_ha
    if forest_area_ha <= 0 or score.valid_count == 0:
        return not_applicable(
            "2.1 Forest Landscape Integrity",
            "The forest integrity layer does not cover the forest in this project area.",
        )

    mean_flii = float(np.ma.mean(score.values))  # 0 to 10, one decimal on display

    rows = tabulate_classes(classes, FLII_CLASSES, denominator_ha=forest_area_ha)
    by_code = {r.code: r for r in rows}
    dom = dominant(rows)

    narrative = sentences(
        f"Of the forest in this area, {fmt_pct(by_code[FLII_HIGH].pct)} has high landscape "
        f"integrity, {fmt_pct(by_code[FLII_MEDIUM].pct)} medium, and "
        f"{fmt_pct(by_code[FLII_LOW].pct)} low.",
        f"The forest is predominantly {dom.label.lower()} integrity, {FLII_GLOSS[dom.code]}.",
    )

    return ComponentResult(
        component="2.1 Forest Landscape Integrity",
        applicable=True,
        narrative=narrative,
        tables={"integrity": rows},
        values={
            "mean_flii": mean_flii,  # headline big number
            "dominant_class": dom.code,
            "pct_high": by_code[FLII_HIGH].pct,
            "forest_area_ha": forest_area_ha,
        },
    )

---
## 2.2 Key Biodiversity Areas (KBA)

Reports whether the AOI overlaps Key Biodiversity Areas, with overlap area, share and a
narrative.

**A KBA is not a protected area.** It is a site that contributes significantly to the global
persistence of biodiversity (IUCN KBA Standard 2016). It may or may not be legally protected.
This is a different lens from 1.3 (WDPA, legal status) and the two complement each other. The
narrative emphasises biodiversity importance, not protection.

**Data.** `KBA_polygon.shp`, World Database of Key Biodiversity Areas, BirdLife International
and the KBA Partnership.

**To verify against the actual file.** Name field assumed here: `IntName`.

**Example render.**

> This project area overlaps 210 ha (17%) of a Key Biodiversity Area, Bukit Tigapuluh. Key
> Biodiversity Areas are sites that contribute significantly to the global persistence of
> biodiversity.

**Downstream use.** Feeds Triple Win Pillar 1 and acts as a safeguard and eligibility signal. A
KBA that is not also under WDPA protection (1.3) is a biodiversity important but unprotected
site, which is a strong PROTECT and additionality rationale.

In [ ]:
KBA_DEFINITION = (
    "Key Biodiversity Areas are sites that contribute significantly to the global persistence "
    "of biodiversity."
)


@dataclass(frozen=True)
class KbaSite:
    name: str
    area_ha: float


def analyze_kba(aoi: AOI) -> ComponentResult:
    """Component 2.2. Overlap with Key Biodiversity Areas."""
    gdf = load_vector_intersecting(KBA_POLYGON, aoi)

    if gdf.empty:
        return ComponentResult(
            component="2.2 Key Biodiversity Areas",
            applicable=True,  # "no overlap" is a real answer, not a missing one
            narrative="This project area does not overlap any Key Biodiversity Areas.",
            tables={"sites": []},
            values={"kba_ha": 0.0, "kba_pct": 0.0, "kba_site_count": 0},
        )

    kba_ha = union_overlap_ha(aoi, gdf)
    kba_pct = safe_pct(kba_ha, aoi.area_ha)

    areas = per_feature_overlap_ha(aoi, gdf)
    sites = sort_by_area([
        KbaSite(name=str(gdf.iloc[i].get("IntName", "Unnamed KBA")), area_ha=float(areas[i]))
        for i in range(len(gdf))
    ])

    if len(sites) == 1:
        head = (
            f"This project area overlaps {fmt_ha(kba_ha)} ({fmt_pct(kba_pct)}) of a Key "
            f"Biodiversity Area, {sites[0].name}."
        )
    else:
        largest, *rest = sites
        rest_text = oxford_join(f"{s.name} ({fmt_ha(s.area_ha)})" for s in rest)
        head = (
            f"This project area overlaps {fmt_ha(kba_ha)} ({fmt_pct(kba_pct)}) of Key "
            f"Biodiversity Areas, across {len(sites)} sites. The largest is {largest.name} "
            f"({fmt_ha(largest.area_ha)}), followed by {rest_text}."
        )

    return ComponentResult(
        component="2.2 Key Biodiversity Areas",
        applicable=True,
        narrative=sentences(head, KBA_DEFINITION),
        tables={"sites": sites},
        values={"kba_ha": kba_ha, "kba_pct": kba_pct, "kba_site_count": len(sites)},
    )

---
## Run and save

In [ ]:
results: dict[str, ComponentResult] = {}

results["2.1"] = analyze_flii(aoi)
results["2.2"] = analyze_kba(aoi)

for key, r in results.items():
    print(f"[{key}] {r.component}{'' if r.applicable else '  (not applicable)'}")
    print(f"      {r.narrative}")
    for f in r.flags:
        print(f"      FLAG: {f}")

In [ ]:
path = save_results(results, aoi, aoi_id, STAGE_NATURE)
print(f"Saved {path}")